# QMI / QMIPWA validation with published $D_s^+\to\pi^-\pi^+\pi^+$ points

This notebook uses the 50 central QMI S-wave points reported in Table 9 of:

LHCb Collaboration, *Amplitude analysis of the $D_s^+\to\pi^-\pi^+\pi^+$ decay*, arXiv:2209.09840.

The published table gives $m_{\pi\pi}$, magnitude and phase. The phase values are quoted in degrees; below they are converted to radians and unwrapped before interpolation. Following the analysis convention, the linear interpolation is performed in $s=m_{\pi\pi}^2$.

The notebook:
1. reproduces the published magnitude, phase and Argand trajectory;
2. builds a $D_s^+\to\pi^-\pi^+\pi^+$ QMI amplitude;
3. generates a toy MC using the published S-wave as truth;
4. demonstrates how the same 50-knot model becomes a 98-parameter fit model by fixing one reference magnitude/phase pair.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DecayChannel,
    DecayModel,
    Parameter,
    QMI,
    RealImag,
    Resonance,
    enable_x64,
    weighted_resample,
)

enable_x64()


## 1. Published 50-point QMI S-wave

In [ ]:
knots = np.array([0.28, 0.39, 0.47, 0.546, 0.623, 0.698, 0.766, 0.819, 0.865, 0.9, 0.925, 0.942, 0.955, 0.964, 0.972, 0.978, 0.983, 0.99, 1.001, 1.023, 1.051, 1.083, 1.116, 1.149, 1.179, 1.205, 1.227, 1.245, 1.261, 1.276, 1.289, 1.302, 1.314, 1.326, 1.338, 1.351, 1.363, 1.375, 1.387, 1.399, 1.411, 1.424, 1.437, 1.45, 1.465, 1.484, 1.511, 1.554, 1.613, 1.823], dtype=float)
published_magnitude = np.array([4.54, 4.05, 4.1, 4.41, 4.69, 4.691, 4.994, 5.43, 6.405, 8.096, 10.624, 13.47, 16.56, 19.45, 22.15, 24.62, 26.95, 20.89, 13.695, 10.995, 9.593, 8.731, 7.606, 6.961, 6.515, 6.506, 6.264, 6.125, 6.081, 6.071, 5.912, 5.893, 5.901, 6.031, 5.904, 6.086, 6.181, 6.185, 6.6, 6.63, 6.9, 7.14, 7.22, 7.56, 7.33, 7.13, 5.009, 2.456, 2.31, 3.75], dtype=float)
published_phase_deg = np.array([176.8, 152.8, 147.6, 146.4, 149.6, 157.4, 169.5, -172.8, -152.0, -133.0, -116.5, -103.0, -88.8, -74.9, -59.5, -56.7, -21.8, -9.8, 5.45, 11.28, 21.57, 36.27, 48.02, 54.42, 63.46, 63.47, 72.45, 72.8, 77.2, 82.2, 86.1, 88.8, 93.2, 93.8, 98.7, 103.3, 105.7, 110.4, 114.5, 119.7, 126.5, 132.3, 142.03, 153.74, 166.5, -172.15, -136.8, -126.8, -99.5, 3.0], dtype=float)

# Preserve the published phases exactly, then unwrap only for interpolation.
published_phase_rad = np.deg2rad(published_phase_deg)
unwrapped_phase_rad = np.unwrap(published_phase_rad)

assert len(knots) == len(published_magnitude) == len(published_phase_deg) == 50

print("number of published QMI points =", len(knots))
print("mass range =", knots[0], "to", knots[-1], "GeV")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)

axes[0].plot(knots, published_magnitude, "o-")
axes[0].set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel="magnitude", title="Published QMI magnitude")

axes[1].plot(knots, published_phase_deg, "o-")
axes[1].set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel="phase [deg]", title="Published QMI phase")

amp_knots = published_magnitude * np.exp(1j * unwrapped_phase_rad)
axes[2].plot(amp_knots.real, amp_knots.imag, "o-")
axes[2].set(xlabel="Re A", ylabel="Im A", title="Published QMI Argand points")
axes[2].set_aspect("equal", adjustable="datalim")

plt.show()


## 2. Continuous QMI amplitude

`QMI` takes the published mass points in GeV but interpolates magnitude and phase linearly as functions of $s=m^2$. The unwrapped phase is used internally to avoid artificial jumps at the $\pm180^\circ$ branch cut.

In [ ]:
qmi_truth = QMI(
    knots=tuple(knots),
    magnitudes=tuple(published_magnitude),
    phases=tuple(unwrapped_phase_rad),
)

m = jnp.linspace(knots[0], knots[-1], 4000)
mag_interp, phase_interp = qmi_truth.interpolated_magnitude_phase(m)
amp_interp = np.asarray(mag_interp * jnp.exp(1j * phase_interp))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)
axes[0].plot(np.asarray(m), np.asarray(mag_interp))
axes[0].plot(knots, published_magnitude, "o")
axes[0].set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel="magnitude")

axes[1].plot(np.asarray(m), np.rad2deg(np.asarray(phase_interp)))
axes[1].plot(knots, np.rad2deg(unwrapped_phase_rad), "o")
axes[1].set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel="unwrapped phase [deg]")

axes[2].plot(amp_interp.real, amp_interp.imag)
axes[2].plot(amp_knots.real, amp_knots.imag, "o")
axes[2].set(xlabel="Re A", ylabel="Im A")
axes[2].set_aspect("equal", adjustable="datalim")
plt.show()


## 3. $D_s^+\to\pi^-\pi^+\pi^+$ model

In [ ]:
channel = DecayChannel("D_s+", ("pi-", "pi+", "pi+"))
owner = "pipi_S_qmi"

truth_model = DecayModel(
    channel,
    [
        Resonance(
            owner,
            pair=(0, 1),
            coefficient=RealImag(1.0, 0.0),
            mass=1.0,
            width=0.0,
            spin=0,
            lineshape=qmi_truth,
        )
    ],
    normalization_resolution=500,
)

print("D_s+ mass =", channel.parent_mass, "GeV")
print("daughter masses =", channel.daughter_masses, "GeV")


## 4. Deterministic Dalitz density

In [ ]:
grid = truth_model.normalization_sample
intensity = np.asarray(truth_model.intensity(grid.as_dict()))
weights = np.asarray(grid.weights) * intensity

fig, ax = plt.subplots(figsize=(7, 6))
h = ax.hist2d(np.asarray(grid.s12), np.asarray(grid.s13), bins=120, weights=weights)
fig.colorbar(h[3], ax=ax, label=r"grid weight $\times |A|^2$")
ax.set(
    xlabel=r"$s_{12}$ [GeV$^2$]",
    ylabel=r"$s_{13}$ [GeV$^2$]",
    title=r"$D_s^+\to(\pi^+\pi^-)_S\pi^+$ — published QMI S-wave",
)
plt.show()

s_pm = np.concatenate([np.asarray(grid.s12), np.asarray(grid.s13)])
w_pm = np.concatenate([weights, weights])
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(s_pm, bins=150, weights=w_pm, histtype="step", linewidth=1.6)
ax.set(
    xlabel=r"$m^2(\pi^+\pi^-)$ [GeV$^2$]",
    ylabel="weighted grid intensity",
    title=r"Published $D_s^+$ QMI S-wave projection",
)
plt.show()


## 5. Toy MC

The MC is used only for proposal/toy generation. Normalization remains deterministic through the Dalitz grid.

In [ ]:
N_POOL = 1_000_000
N_TOY = 100_000

pool = truth_model.generate_phase_space(N_POOL, seed=5200)
pool_intensity = truth_model.intensity(pool.as_dict())
target_weights = pool.weights * pool_intensity

toy = weighted_resample(
    jax.random.key(5201),
    pool,
    target_weights,
    N_TOY,
    replace=True,
)

fig, ax = plt.subplots(figsize=(7, 6))
h = ax.hist2d(np.asarray(toy.s12), np.asarray(toy.s13), bins=100)
fig.colorbar(h[3], ax=ax, label="events")
ax.set(
    xlabel=r"$s_{12}$ [GeV$^2$]",
    ylabel=r"$s_{13}$ [GeV$^2$]",
    title=r"$D_s^+\to\pi^-\pi^+\pi^+$ QMI toy — 100k events",
)
plt.show()

s_toy = np.concatenate([np.asarray(toy.s12), np.asarray(toy.s13)])
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(s_toy, bins=150, histtype="step", linewidth=1.6)
ax.set(
    xlabel=r"$m^2(\pi^+\pi^-)$ [GeV$^2$]",
    ylabel="entries / bin",
    title=r"QMI toy $s_{12}+s_{13}$ projection",
)
plt.show()


## 6. Full 50-knot fit parameterisation

A QMI-only component has an arbitrary overall complex scale. For a fit we therefore fix one knot's magnitude and phase and let the remaining 49 magnitudes and 49 phases float. Here the starting values are the published central values themselves; a real closure test can randomize them before minimization.

In [ ]:
fit_magnitudes = [float(published_magnitude[0])]
fit_phases = [float(unwrapped_phase_rad[0])]

for i in range(1, len(knots)):
    fit_magnitudes.append(
        Parameter.dynamics(
            f"qmi.a{i}",
            float(published_magnitude[i]),
            owner=owner,
            bounds=(0.0, None),
            step=0.02,
        )
    )
    fit_phases.append(
        Parameter.dynamics(
            f"qmi.d{i}",
            float(unwrapped_phase_rad[i]),
            owner=owner,
            step=0.03,
        )
    )

qmi_fit = QMI(
    knots=tuple(knots),
    magnitudes=tuple(fit_magnitudes),
    phases=tuple(fit_phases),
)

fit_model = DecayModel(
    channel,
    [
        Resonance(
            owner,
            pair=(0, 1),
            coefficient=RealImag(1.0, 0.0),
            mass=1.0,
            width=0.0,
            spin=0,
            lineshape=qmi_fit,
        )
    ],
    normalization_resolution=500,
)

print("free QMI parameters =", len(fit_model.parameters))
print("expected =", 2 * (len(knots) - 1))
